 Controlling for Confounding Variables (Apples-to-Apples Comparison)
Provinces may naturally have different distributions of client ages, vehicle types, or insurance plans. To ensure that any observed risk variation is truly caused by the *geographical region* and not by asset or demographic imbalances, we will:
1. Identify the most dominant, baseline profile in the dataset (e.g., standard vehicle type, average age bracket).
2. Subset the data to include only this controlled profile.
3. Define **Group A (Control)** as Province 1 and **Group B (Test)** as Province 2 within this controlled subset.

In [7]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import chi2_contingency

df = pd.read_csv(r"C:\Users\hp\insurance-risk-analytics\data\insurance_data.csv")

# 2. Check which provinces have the most data to choose our Group A and B
print("Top provinces in dataset:")
print(df['Province'].value_counts().head(5))

# 3. Find the most common profile to keep the comparison fair
print("\nMost common vehicle types:")
print(df['VehicleType'].value_counts().head(3))


Top provinces in dataset:
Province
Addis Ababa    3567
Oromia         2446
Amhara         1999
Somali         1184
Tigray          804
Name: count, dtype: int64

Most common vehicle types:
VehicleType
Sedan        3992
SUV          3000
Hatchback    2036
Name: count, dtype: int64


**Step 1: Filter our Data**

In [8]:
# Filter the dataset for our controlled baseline profile (Sedans only)
sedan_df = df[df['VehicleType'] == 'Sedan']

# Segment into Group A (Addis Ababa) and Group B (Oromia)
group_A = sedan_df[sedan_df['Province'] == 'Addis Ababa']
group_B = sedan_df[sedan_df['Province'] == 'Oromia']

print(f"Group A (Addis Ababa Sedans) Sample Size: {len(group_A)}")
print(f"Group B (Oromia Sedans) Sample Size: {len(group_B)}")

Group A (Addis Ababa Sedans) Sample Size: 1421
Group B (Oromia Sedans) Sample Size: 992


**Step 2: Test Part 1 — Claim Frequency (Chi-Squared Test)**

In [11]:
print("\n--- Running Chi-Squared Test for Claim Frequency ---")

# Create the contingency table (counts of Claims vs No Claims for both regions)

contingency_table = pd.crosstab(sedan_df[sedan_df['Province'].isin(['Addis Ababa', 'Oromia'])]['Province'], 
                                sedan_df['Claimed'])

print("Contingency Table:")
print(contingency_table)

# Run the test
chi2, p_val_freq, dof, expected = chi2_contingency(contingency_table)
print(f"Chi-Squared P-value: {p_val_freq:.5f}")

if p_val_freq < 0.05:
    print("Result: Reject H0. There is a significant difference in Claim Frequency between provinces.")
else:
    print("Result: Fail to Reject H0. No significant difference in Claim Frequency.")


--- Running Chi-Squared Test for Claim Frequency ---
Contingency Table:
Claimed      False  True 
Province                 
Addis Ababa   1233    188
Oromia         863    129
Chi-Squared P-value: 0.91993
Result: Fail to Reject H0. No significant difference in Claim Frequency.


### Business Interpretation: Province Risk Analysis (Claim Frequency)

* **Statistical Result:** Fail to Reject $H_0$ ($p = 0.920$)
* **Key Finding:** There is **no statistically significant difference** in claim frequency between **Addis Ababa** and **Oromia** for sedan vehicles. 

#### Business Context & Recommendation
The data shows that approximately **13.2%** of sedan policies in Addis Ababa filed a claim ($188 / 1421$), compared to **13.0%** of sedan policies in Oromia ($129 / 992$). 

Because the $p$-value ($0.920$) is well above our significance threshold of $0.05$, this minor $0.2\%$ difference is entirely due to random baseline noise rather than actual geographic risk factors. 

**Actionable Insight:** ACIS should **not** implement different baseline premium prices for sedan policies based on these provinces alone, as geographic location does not currently act as a differentiator for how *frequently* accidents occur here. We should keep premium structures unified across these regions for this segment unless a variance is discovered in claim severity.

In [12]:
print("\n--- Running T-Test for Claim Severity ---")

# Filter for only those who filed a claim and compare the ClaimAmount
group_A_severity = group_A[group_A['ClaimAmount'] > 0]['ClaimAmount']
group_B_severity = group_B[group_B['ClaimAmount'] > 0]['ClaimAmount']

print(f"Average Claim Cost in Addis Ababa: {group_A_severity.mean():.2f}")
print(f"Average Claim Cost in Oromia: {group_B_severity.mean():.2f}")

# Run Welch's T-Test (equal_var=False)
t_stat, p_val_sev = stats.ttest_ind(group_A_severity, group_B_severity, equal_var=False)
print(f"T-Test P-value: {p_val_sev:.5f}")

if p_val_sev < 0.05:
    print("Result: Reject H0. There is a significant difference in Claim Severity between provinces.")
else:
    print("Result: Fail to Reject H0. No significant difference in Claim Severity.")


--- Running T-Test for Claim Severity ---
Average Claim Cost in Addis Ababa: 6809.77
Average Claim Cost in Oromia: 7314.34
T-Test P-value: 0.38406
Result: Fail to Reject H0. No significant difference in Claim Severity.


### Business Interpretation: Province Risk Analysis (Claim Severity)

* **Statistical Result:** Fail to Reject $H_0$ ($p = 0.384$)
* **Key Finding:** There is **no statistically significant difference** in claim severity (the financial cost of a claim) between **Addis Ababa** and **Oromia** for sedan vehicles.

#### Business Context & Recommendation
While the raw sample data indicates that the average claim cost in Oromia (7,314.34) was roughly 504.57 units higher than in Addis Ababa (6,809.77), the high $p$-value ($0.384$) demonstrates that this gap is not statistically dependable. In small or variance-heavy insurance samples, fluctuations of this size are common due to random chance (e.g., a couple of unusually expensive repairs in one region).

**Actionable Insight:** Combined with our claim frequency findings, we have verified that geography across these primary provinces does not significantly shift overall underwriting risk for sedans. ACIS should **maintain uniform base pricing and claim cost expectations** across Addis Ababa and Oromia rather than introducing regional premium adjustments.

# Hypothesis 2: Risk Analysis Across Zip Codes

## 1. Objective
To investigate whether hyper-local geographic factors influence insurance risk by evaluating differences in **Claim Frequency** and **Claim Severity** across different zip codes.

## 2. Hypotheses Defined
* **Null Hypothesis ($H_0$):** There are no risk differences between zip codes. (The specific zip code does not affect claims).
* **Alternative Hypothesis ($H_1$):** There is a statistically significant difference in risk between different zip codes.

In [14]:
print(df['ZipCode'].value_counts().head(5))

ZipCode
10004    733
10002    732
10003    714
10001    710
10005    678
Name: count, dtype: int64


In [34]:
import pandas as pd
from scipy import stats
from scipy.stats import chi2_contingency

# FIX: Use raw integers instead of strings with quotes
target_zips = [10004, 10002]

# 1. Filter the rows for just the two target zip codes
zip_subset = df[df['ZipCode'].isin(target_zips)]

print(f"Total rows found for these zip codes: {len(zip_subset)}")

# 2. Find the most common vehicle type within those zip codes safely
if len(zip_subset) > 0:
    most_common_vehicle = zip_subset['VehicleType'].mode().iloc[0]
    print(f"Most common vehicle type in these zip codes: {most_common_vehicle}")

    # Keep only rows with that vehicle type
    filtered_zip_df = zip_subset[zip_subset['VehicleType'] == most_common_vehicle]
    
    # Split into explicit control and test groups using integer values
    group_A_zip = filtered_zip_df[filtered_zip_df['ZipCode'] == 10004]
    group_B_zip = filtered_zip_df[filtered_zip_df['ZipCode'] == 10002]
    
    # ==========================================
    # PART 1: CLAIM FREQUENCY (Chi-Squared Test)
    # ==========================================
    print("\n--- Running Chi-Squared Test for Zip Code Claim Frequency ---")
    contingency_table_zip = pd.crosstab(filtered_zip_df['ZipCode'], filtered_zip_df['Claimed'])
    print("Contingency Table:")
    print(contingency_table_zip)

    chi2, p_val_zip_freq, dof, expected = chi2_contingency(contingency_table_zip)
    print(f"Chi-Squared P-value: {p_val_zip_freq:.5f}")

    # ==========================================
    # PART 2: CLAIM SEVERITY (T-Test)
    # ==========================================
    print("\n--- Running T-Test for Zip Code Claim Severity ---")
    group_A_zip_sev = group_A_zip[group_A_zip['ClaimAmount'] > 0]['ClaimAmount']
    group_B_zip_sev = group_B_zip[group_B_zip['ClaimAmount'] > 0]['ClaimAmount']

    if len(group_A_zip_sev) > 1 and len(group_B_zip_sev) > 1:
        print(f"Average Claim Cost in Zip 10004: {group_A_zip_sev.mean():.2f}")
        print(f"Average Claim Cost in Zip 10002: {group_B_zip_sev.mean():.2f}")
        t_stat, p_val_zip_sev = stats.ttest_ind(group_A_zip_sev, group_B_zip_sev, equal_var=False)
        print(f"T-Test P-value: {p_val_zip_sev:.5f}")
    else:
        print("Not enough claim events in these subsets to run a reliable Severity T-test.")

Total rows found for these zip codes: 1465
Most common vehicle type in these zip codes: Sedan

--- Running Chi-Squared Test for Zip Code Claim Frequency ---
Contingency Table:
Claimed  False  True 
ZipCode              
10002      258     45
10004      254     36
Chi-Squared P-value: 0.45660

--- Running T-Test for Zip Code Claim Severity ---
Average Claim Cost in Zip 10004: 6586.97
Average Claim Cost in Zip 10002: 6690.89
T-Test P-value: 0.89151


### Statistical Analysis & Business Interpretation: Zip Code Risk

* **Claim Frequency Result:** Fail to Reject $H_0$ ($p = 0.457$)
* **Claim Severity Result:** Fail to Reject $H_0$ ($p = 0.892$)
* **Key Finding:** There are **no statistically significant risk differences** between Zip Code 10004 and Zip Code 10002 for sedan vehicles.

#### Direct Data Comparison
| Metric | Zip Code 10004 (Control) | Zip Code 10002 (Test) | Absolute Difference | Statistical Significance |
| :--- | :---: | :---: | :---: | :---: |
| **Claim Frequency (%)** | 12.4% | 14.8% | +2.4% | No ($p = 0.457$) |
| **Avg Claim Severity** | 6,586.97 | 6,690.89 | +103.92 | No ($p = 0.892$) |

#### Business Context & Actionable Recommendations

1. **Maintain Unified Pricing Structure:** The minor risk elevations observed in Zip Code 10002 are entirely due to normal random fluctuations in underwriting data. Therefore, ACIS should **not** split these regions into different pricing tiers or charge higher localized base premiums for Zip Code 10002. Dropping both zip codes into the same baseline geographic risk pool is mathematically sound.
2. **Optimize Operational Allocation:** Because both the probability of an accident occurring and the actual repair cost demands are equivalent across these areas, claims management teams and emergency response networks can expect identical per-policy operational burdens in both territories. 
3. **Strategic Next Steps:** Since macro-geography (Provinces) and micro-geography (Zip Codes) have failed to show risk variances for sedans, regional demographics are proving to be neutral risk drivers. We recommend focusing future analysis on behavioral and client-specific indicators (such as driver age or policy tier) to locate true risk differentiation for ACIS's segmentation strategy.

# Hypothesis 3: Profit Margin Analysis Across Zip Codes

## 1. Objective
To determine if specific hyper-local geographic zones yield significantly higher or lower financial returns for ACIS. This helps identify whether certain zip codes are highly profitable or drains on corporate capital.

## 2. Hypotheses Defined
* **Null Hypothesis ($H_0$):** There is no significant margin (profit) difference between zip codes. (The expected profit per policy is uniform across locations).
* **Alternative Hypothesis ($H_1$):** There is a statistically significant difference in profit margins between zip codes.

## 3. Metric Definition
* **Margin (Profit):** Calculated on a per-policy basis as:
  $$\text{Margin} = \text{TotalPremium} - \text{TotalClaims}$$
* **Methodology:** Since Margin is a continuous, numerical value, we will use a **Two-Sample Independent T-Test (Welch's T-Test)** to evaluate the mean profit difference between our target groups.

In [35]:
import pandas as pd
from scipy import stats

# 1. Filter the dataset for Sedans in our target zip codes
zip_margin_df = df[(df['VehicleType'] == 'Sedan') & (df['ZipCode'].isin([10004, 10002]))].copy()

zip_margin_df['Margin'] = zip_margin_df['TotalPremium'] - zip_margin_df['TotalClaims']

# 3. Split into our distinct control and test groups
group_A_margin = zip_margin_df[zip_margin_df['ZipCode'] == 10004]['Margin']
group_B_margin = zip_margin_df[zip_margin_df['ZipCode'] == 10002]['Margin']

print("--- Running T-Test for Zip Code Profit Margin ---")
print(f"Average Profit Margin per Policy in Zip 10004: {group_A_margin.mean():.2f}")
print(f"Average Profit Margin per Policy in Zip 10002: {group_B_margin.mean():.2f}")

# 4. Run Welch's T-Test on the profit distributions
t_stat_margin, p_val_margin = stats.ttest_ind(group_A_margin, group_B_margin, equal_var=False)
print(f"T-Test P-value for Margin: {p_val_margin:.5f}")

if p_val_margin < 0.05:
    print("Result: Reject H0. There is a significant difference in profit margins between these zip codes.")
else:
    print("Result: Fail to Reject H0. No significant difference in profit margins.")

--- Running T-Test for Zip Code Profit Margin ---
Average Profit Margin per Policy in Zip 10004: 1388.76
Average Profit Margin per Policy in Zip 10002: 1228.49
T-Test P-value for Margin: 0.43384
Result: Fail to Reject H0. No significant difference in profit margins.


### Statistical Analysis & Business Interpretation: Profit Margin

* **Statistical Result:** Fail to Reject $H_0$ ($p = 0.434$)
* **Key Finding:** There is **no statistically significant difference** in the net profit margins generated between Zip Code 10004 and Zip Code 10002 for sedan vehicles.

#### Direct Margin Comparison
| Metric | Zip Code 10004 (Control) | Zip Code 10002 (Test) | Absolute Difference | Statistical Significance |
| :--- | :---: | :---: | :---: | :---: |
| **Avg Profit Margin** | 1,388.76 | 1,228.49 | -160.27 | No ($p = 0.434$) |

#### Business Context & Actionable Recommendations

1. **Equitable Value Creation:** This test proves that the financial yield ACIS extracts per policy is stable across these two primary zip codes. Even though Zip Code 10002 had a slightly higher raw claim rate and cost (from our previous test), its premium income successfully absorbs that exposure, resulting in a bottom-line profit margin that is statistically identical to Zip Code 10004.
2. **Marketing and Sales Strategy:** Since both zip codes generate equivalent profit margins, sales campaigns and growth investments do not need to favor one over the other. ACIS can confidently scale up its marketing footprint in both territories knowing that a sedan policy written in Zip 10002 is worth just as much to the bottom line as one written in Zip 10004.
3. **Geographic Consolidation Summary:** Across all three geographic evaluations (Provinces, Zip Code Risk, and Zip Code Margin), we have consistently failed to reject the null hypothesis. Geography is **not** a primary driver of risk or profitability variance for this segment. We can officially advise the executive team that a unified, non-regional pricing model is completely safe to maintain for these territories.

# Hypothesis 4: Demographic Risk Analysis (Gender)

## 1. Objective
To evaluate if gender acts as a statistically verifiable driver of insurance risk at ACIS by analyzing differences in **Claim Frequency** and **Claim Severity** between male and female policyholders.

## 2. Hypotheses Defined
* **Null Hypothesis ($H_0$):** There is no significant risk difference between Women and Men.
* **Alternative Hypothesis ($H_1$):** There is a statistically significant risk difference between Women and Men.

## 3. Methodology
To ensure a clean comparison, we will isolate our most dominant vehicle segment (**Sedans**) to eliminate asset-class confounding. We will run:
1. A **Chi-Squared Test** on Claim Frequency (Categorical).
2. A **Two-Sample Welch's T-Test** on Claim Severity (Numerical).

In [37]:
import pandas as pd
from scipy import stats
from scipy.stats import chi2_contingency

# 1. Filter for our standard profile (Sedans)
gender_df = df[df['VehicleType'] == 'Sedan'].copy()

# Inspect how gender is recorded in your dataset
print("Gender value counts in your sedan data:")
print(gender_df['Gender'].value_counts())

# 2. Split into explicit Group A and Group B
group_M_gender = gender_df[gender_df['Gender'] == 'Male']
group_F_gender = gender_df[gender_df['Gender'] == 'Female']

# ==========================================
# PART 1: CLAIM FREQUENCY (Chi-Squared Test)
# ==========================================
print("\n--- Running Chi-Squared Test for Gender Claim Frequency ---")
contingency_table_gender = pd.crosstab(gender_df['Gender'], gender_df['Claimed'])
print("Contingency Table:")
print(contingency_table_gender)

chi2, p_val_gen_freq, dof, expected = chi2_contingency(contingency_table_gender)
print(f"Chi-Squared P-value for Gender: {p_val_gen_freq:.5f}")


# ==========================================
# PART 2: CLAIM SEVERITY (T-Test)
# ==========================================
print("\n--- Running T-Test for Gender Claim Severity ---")
group_M_gen_sev = group_M_gender[group_M_gender['ClaimAmount'] > 0]['ClaimAmount']
group_F_gen_sev = group_F_gender[group_F_gender['ClaimAmount'] > 0]['ClaimAmount']

if len(group_M_gen_sev) > 1 and len(group_F_gen_sev) > 1:
    print(f"Average Claim Cost for Men: {group_M_gen_sev.mean():.2f}")
    print(f"Average Claim Cost for Women: {group_F_gen_sev.mean():.2f}")
    t_stat, p_val_gen_sev = stats.ttest_ind(group_M_gen_sev, group_F_gen_sev, equal_var=False)
    print(f"T-Test P-value for Gender Severity: {p_val_gen_sev:.5f}")
else:
    print("Not enough claim events to run a reliable Severity T-test.")

Gender value counts in your sedan data:
Gender
Female    2069
Male      1923
Name: count, dtype: int64

--- Running Chi-Squared Test for Gender Claim Frequency ---
Contingency Table:
Claimed  False  True 
Gender               
Female    1793    276
Male      1681    242
Chi-Squared P-value for Gender: 0.50770

--- Running T-Test for Gender Claim Severity ---
Average Claim Cost for Men: 6985.34
Average Claim Cost for Women: 6838.14
T-Test P-value for Gender Severity: 0.71437


### Statistical Analysis & Business Interpretation: Gender Risk Profile

* **Claim Frequency Result:** Fail to Reject $H_0$ ($p = 0.50770$)
* **Claim Severity Result:** Fail to Reject $H_0$ ($p = 0.71437$)
* **Key Finding:** There is **no statistically significant risk difference** between male and female drivers for sedan vehicles in this dataset.

#### Direct Data Comparison
| Metric | Female Profile | Male Profile | Absolute Difference | Statistical Significance |
| :--- | :---: | :---: | :---: | :---: |
| **Sample Size (Sedans)** | 2,069 | 1,923 | +146 policies (F) | N/A |
| **Claim Frequency (%)** | 13.34% | 12.58% | +0.76% (F) | No ($p = 0.508$) |
| **Avg Claim Severity** | 6,838.14 | 6,985.34 | +147.20 (M) | No ($p = 0.714$) |

#### Business Context & Actionable Recommendations

1. **Implement Gender-Neutral Base Pricing:** The statistical analysis confirms that gender does not structurally alter underwriting risk for sedans at ACIS. The minor variations in accident rates and repair costs are mathematically uninformative noise. We recommend maintaining a **fully unified, gender-neutral premium pricing model** for this vehicle tier.
2. **Compliance & Brand Integrity:** Aligning the core pricing model to these findings ensures absolute transparency. It safeguards ACIS from demographic bias claims, demonstrating that company adjustments are driven entirely by statistically verifiable risk variables rather than broad assumptions.
3. **Strategic Analytical Pivot:** Since traditional demographic splits (Geography, Gender) are returning neutral risk variances, the risk engine should pivot toward behavioral indicators. Future sprints should explore features such as **Driver Age distributions, Policy Tier levels, or Vehicle Age** to find actionable risk clusters.